# Popular Times Trend Analysis with Silhouette Method

This notebook analyzes popular times patterns across different stores and uses the silhouette method to identify optimal clusters (trends) in the data.

## Objectives:
1. Load and preprocess popular times data
2. Correlate popular times patterns across stores
3. Use silhouette analysis to determine optimal number of clusters
4. Visualize trends and patterns

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
import warnings
import os
import sys

warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 8)

print("Libraries imported successfully!")
print(f"Project root: {project_root}")

## 1. Data Loading and Preprocessing

In [ ]:
# Sample data structure provided by user
# Create a sample dataset or load from CSV if available

# For demonstration, you can either:
# 1. Load from a CSV file if you have it
# 2. Use the sample data provided

# Example: Load from CSV (uncomment if you have the file)
# data_path = os.path.join(project_root, 'data', 'popular_times.csv')
# df = pd.read_csv(data_path)

# For now, let's create a function to load the data
def load_popular_times_data(file_path=None):
    """
    Load popular times data from CSV file.
    If no file is provided, use sample data.
    
    Expected columns: hour, Reference_Id, Location, Day, Ratio, Adjusted_ratio
    """
    if file_path and os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"Data loaded from {file_path}")
    else:
        print("Please provide the data file path or paste your data into a CSV file.")
        print("Expected columns: hour, Reference_Id, Location, Day, Ratio, Adjusted_ratio")
        # You can manually create a CSV file in the data directory with your data
        return None
    
    return df

# Try to load the data
data_path = os.path.join(project_root, 'data', 'popular_times.csv')
df = load_popular_times_data(data_path)

if df is not None:
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    display(df.head())
    
    print(f"\nUnique stores: {df['Reference_Id'].nunique()}")
    print(f"Unique days: {df['Day'].unique()}")
    print(f"\nData types:\n{df.dtypes}")
else:
    print("\n⚠️  Please create a CSV file at 'data/popular_times.csv' with your popular times data.")
    print("   The file should contain columns: hour, Reference_Id, Location, Day, Ratio, Adjusted_ratio")

In [ ]:
# Data preprocessing and exploration
if df is not None:
    # Check for missing values
    print("Missing values:")
    print(df.isnull().sum())
    
    # Basic statistics
    print("\nBasic statistics:")
    display(df.describe())
    
    # Check data completeness for each store
    print("\nData completeness check:")
    records_per_store = df.groupby('Reference_Id').size()
    print(f"Expected records per store: {24 * 7} (24 hours × 7 days)")
    print(f"Actual records per store:")
    print(records_per_store.value_counts())

## 2. Feature Engineering for Clustering

We'll create features that capture the popular times patterns for each store.

In [ ]:
def create_store_features(df):
    """
    Create feature matrix for clustering.
    Each store is represented by its hourly patterns across all days.
    """
    # Pivot the data to have stores as rows and (day, hour) combinations as columns
    # Create a combined day-hour column
    df['day_hour'] = df['Day'] + '_' + df['hour'].astype(str)
    
    # Pivot to get features: one column for each day-hour combination
    feature_matrix = df.pivot_table(
        index='Reference_Id',
        columns='day_hour',
        values='Adjusted_ratio',
        aggfunc='first'
    )
    
    # Alternative: Just use hourly patterns averaged across all days
    hourly_features = df.pivot_table(
        index='Reference_Id',
        columns='hour',
        values='Adjusted_ratio',
        aggfunc='mean'
    )
    
    # Store location mapping
    location_map = df.groupby('Reference_Id')['Location'].first()
    
    return feature_matrix, hourly_features, location_map

if df is not None:
    feature_matrix, hourly_features, location_map = create_store_features(df)
    
    print(f"Feature matrix shape: {feature_matrix.shape}")
    print(f"Hourly features shape: {hourly_features.shape}")
    print(f"\nFeature matrix (first few rows):")
    display(feature_matrix.head())
    
    print(f"\nHourly features (averaged across days):")
    display(hourly_features.head())

## 3. Silhouette Analysis for Optimal Cluster Determination

The silhouette method helps us determine the optimal number of clusters by measuring how similar an object is to its own cluster compared to other clusters.

In [ ]:
def perform_silhouette_analysis(X, max_clusters=10):
    """
    Perform silhouette analysis for different numbers of clusters.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix (scaled)
    max_clusters : int
        Maximum number of clusters to test
    
    Returns:
    --------
    dict with silhouette scores and optimal k
    """
    silhouette_scores = []
    inertias = []
    K_range = range(2, max_clusters + 1)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        
        # Calculate silhouette score
        score = silhouette_score(X, labels)
        silhouette_scores.append(score)
        
        # Calculate inertia (for elbow method comparison)
        inertias.append(kmeans.inertia_)
        
        print(f"k={k}: Silhouette Score = {score:.4f}, Inertia = {kmeans.inertia_:.4f}")
    
    # Find optimal k (highest silhouette score)
    optimal_k = K_range[np.argmax(silhouette_scores)]
    
    return {
        'silhouette_scores': silhouette_scores,
        'inertias': inertias,
        'K_range': list(K_range),
        'optimal_k': optimal_k,
        'optimal_score': max(silhouette_scores)
    }

if df is not None:
    # Use hourly features (simpler and more interpretable)
    X = hourly_features.fillna(0).values
    
    # Standardize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"Scaled feature matrix shape: {X_scaled.shape}\n")
    print("Performing silhouette analysis...\n")
    print("=" * 60)
    
    # Determine max clusters (not more than n_samples - 1)
    max_clusters = min(10, X_scaled.shape[0] - 1)
    
    silhouette_results = perform_silhouette_analysis(X_scaled, max_clusters=max_clusters)
    
    print("=" * 60)
    print(f"\n✓ Optimal number of clusters: {silhouette_results['optimal_k']}")
    print(f"✓ Best silhouette score: {silhouette_results['optimal_score']:.4f}")

In [ ]:
# Visualize silhouette scores and elbow curve
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Silhouette score plot
    axes[0].plot(silhouette_results['K_range'], 
                 silhouette_results['silhouette_scores'], 
                 'bo-', linewidth=2, markersize=8)
    axes[0].axvline(x=silhouette_results['optimal_k'], 
                    color='r', linestyle='--', linewidth=2,
                    label=f"Optimal k={silhouette_results['optimal_k']}")
    axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
    axes[0].set_ylabel('Silhouette Score', fontsize=12)
    axes[0].set_title('Silhouette Score vs Number of Clusters', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Elbow curve (inertia)
    axes[1].plot(silhouette_results['K_range'], 
                 silhouette_results['inertias'], 
                 'go-', linewidth=2, markersize=8)
    axes[1].axvline(x=silhouette_results['optimal_k'], 
                    color='r', linestyle='--', linewidth=2,
                    label=f"Optimal k={silhouette_results['optimal_k']}")
    axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
    axes[1].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
    axes[1].set_title('Elbow Curve - Inertia vs Number of Clusters', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Interpretation:")
    print("   - Higher silhouette score indicates better-defined clusters")
    print("   - Silhouette score ranges from -1 to 1")
    print("   - Score > 0.5: Strong cluster structure")
    print("   - Score 0.3-0.5: Weak cluster structure")
    print("   - Score < 0.3: No substantial structure")

## 4. Detailed Silhouette Analysis for Optimal K

Let's examine the silhouette plot for the optimal number of clusters.

In [ ]:
def plot_silhouette_analysis(X, n_clusters):
    """
    Create a detailed silhouette plot for a given number of clusters.
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    
    # Perform clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    
    # Calculate silhouette scores
    silhouette_avg = silhouette_score(X, cluster_labels)
    sample_silhouette_values = silhouette_samples(X, cluster_labels)
    
    y_lower = 10
    colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))
    
    for i in range(n_clusters):
        # Aggregate silhouette scores for samples in cluster i
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()
        
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, ith_cluster_silhouette_values,
                         facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
        
        # Label the silhouette plots with their cluster numbers
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), 
                fontsize=12, fontweight='bold')
        
        y_lower = y_upper + 10
    
    ax.set_title(f'Silhouette Plot for {n_clusters} Clusters', fontsize=14, fontweight='bold')
    ax.set_xlabel('Silhouette Coefficient Values', fontsize=12)
    ax.set_ylabel('Cluster Label', fontsize=12)
    
    # Draw vertical line for average silhouette score
    ax.axvline(x=silhouette_avg, color="red", linestyle="--", linewidth=2,
               label=f'Average Score: {silhouette_avg:.3f}')
    ax.legend(fontsize=11)
    
    ax.set_yticks([])
    ax.set_xlim([-0.1, 1])
    
    plt.tight_layout()
    plt.show()
    
    return cluster_labels, silhouette_avg

if df is not None:
    optimal_k = silhouette_results['optimal_k']
    cluster_labels, avg_score = plot_silhouette_analysis(X_scaled, optimal_k)
    
    print(f"\n📊 Silhouette Analysis for k={optimal_k}:")
    print(f"   Average silhouette score: {avg_score:.4f}")
    print("\n   The thickness of each cluster silhouette represents the cluster size.")
    print("   Silhouette coefficients closer to 1 indicate well-separated clusters.")

## 5. Cluster Analysis and Visualization

In [ ]:
# Perform final clustering with optimal k
if df is not None:
    optimal_k = silhouette_results['optimal_k']
    
    # Final clustering
    kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    final_labels = kmeans_final.fit_predict(X_scaled)
    
    # Add cluster labels to the dataframe
    hourly_features['cluster'] = final_labels
    
    # Show cluster distribution
    print("Cluster Distribution:")
    cluster_counts = pd.Series(final_labels).value_counts().sort_index()
    print(cluster_counts)
    
    # Show stores in each cluster
    print("\nStores by Cluster:")
    print("=" * 80)
    for cluster_id in range(optimal_k):
        store_ids = hourly_features[hourly_features['cluster'] == cluster_id].index
        print(f"\nCluster {cluster_id} ({len(store_ids)} stores):")
        for store_id in store_ids:
            location = location_map.get(store_id, 'Unknown')
            print(f"  - Store {store_id}: {location}")

In [ ]:
# Visualize cluster patterns (hourly trends)
if df is not None:
    fig, axes = plt.subplots(optimal_k, 1, figsize=(15, 5 * optimal_k))
    
    if optimal_k == 1:
        axes = [axes]
    
    colors = plt.cm.tab10(np.arange(optimal_k))
    
    for cluster_id in range(optimal_k):
        # Get stores in this cluster
        cluster_stores = hourly_features[hourly_features['cluster'] == cluster_id]
        
        # Plot individual store patterns
        for idx, (store_id, row) in enumerate(cluster_stores.iterrows()):
            hourly_values = row.drop('cluster').values
            axes[cluster_id].plot(range(24), hourly_values, 
                                 alpha=0.3, linewidth=1, color='gray')
        
        # Plot cluster centroid
        cluster_mean = cluster_stores.drop('cluster', axis=1).mean()
        axes[cluster_id].plot(range(24), cluster_mean.values, 
                             linewidth=3, color=colors[cluster_id],
                             label=f'Cluster {cluster_id} Average', marker='o')
        
        axes[cluster_id].set_xlabel('Hour of Day', fontsize=11)
        axes[cluster_id].set_ylabel('Adjusted Ratio', fontsize=11)
        axes[cluster_id].set_title(f'Cluster {cluster_id} - Popular Times Pattern ({len(cluster_stores)} stores)', 
                                  fontsize=13, fontweight='bold')
        axes[cluster_id].set_xticks(range(0, 24, 2))
        axes[cluster_id].legend(fontsize=10)
        axes[cluster_id].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📈 Trend Patterns:")
    print("   Gray lines represent individual stores")
    print("   Colored lines represent the average pattern for each cluster")

## 6. Cluster Characterization

Let's analyze the key characteristics of each cluster.

In [ ]:
# Characterize each cluster
if df is not None:
    print("Cluster Characteristics:")
    print("=" * 80)
    
    for cluster_id in range(optimal_k):
        cluster_data = hourly_features[hourly_features['cluster'] == cluster_id].drop('cluster', axis=1)
        
        # Calculate statistics
        mean_pattern = cluster_data.mean()
        peak_hour = mean_pattern.idxmax()
        peak_value = mean_pattern.max()
        valley_hour = mean_pattern.idxmin()
        valley_value = mean_pattern.min()
        
        print(f"\nCluster {cluster_id}:")
        print(f"  • Number of stores: {len(cluster_data)}")
        print(f"  • Peak hour: {peak_hour}:00 (ratio: {peak_value:.3f})")
        print(f"  • Lowest hour: {valley_hour}:00 (ratio: {valley_value:.3f})")
        print(f"  • Average ratio: {mean_pattern.mean():.3f}")
        print(f"  • Variability (std): {mean_pattern.std():.3f}")
        
        # Identify pattern type
        morning_avg = mean_pattern[6:12].mean()
        afternoon_avg = mean_pattern[12:18].mean()
        evening_avg = mean_pattern[18:24].mean()
        night_avg = mean_pattern[0:6].mean()
        
        peak_period = max(
            [('Morning (6-12)', morning_avg),
             ('Afternoon (12-18)', afternoon_avg),
             ('Evening (18-24)', evening_avg),
             ('Night (0-6)', night_avg)],
            key=lambda x: x[1]
        )
        
        print(f"  • Peak period: {peak_period[0]}")

## 7. PCA Visualization

Visualize clusters in 2D space using Principal Component Analysis (PCA).

In [ ]:
# PCA visualization
if df is not None:
    # Apply PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # Create scatter plot
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.arange(optimal_k))
    
    for cluster_id in range(optimal_k):
        mask = final_labels == cluster_id
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=[colors[cluster_id]], 
                   label=f'Cluster {cluster_id}',
                   s=200, alpha=0.6, edgecolors='black', linewidth=1.5)
        
        # Add store labels
        for idx, store_id in enumerate(hourly_features[hourly_features['cluster'] == cluster_id].index):
            point_idx = list(hourly_features.index).index(store_id)
            plt.annotate(str(store_id), 
                        (X_pca[point_idx, 0], X_pca[point_idx, 1]),
                        fontsize=9, ha='center', va='center')
    
    plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
    plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
    plt.title('Store Clusters in PCA Space', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nPCA Explained Variance:")
    print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
    print(f"  Total: {sum(pca.explained_variance_ratio_):.2%}")

## 8. Correlation Analysis

Analyze correlations between stores within and across clusters.

In [ ]:
# Correlation heatmap
if df is not None:
    # Calculate correlation matrix
    correlation_matrix = hourly_features.drop('cluster', axis=1).T.corr()
    
    # Sort by cluster for better visualization
    sorted_indices = hourly_features.sort_values('cluster').index
    correlation_matrix_sorted = correlation_matrix.loc[sorted_indices, sorted_indices]
    
    # Plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix_sorted, 
                annot=True if len(sorted_indices) <= 10 else False,
                cmap='coolwarm', 
                center=0,
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8},
                fmt='.2f')
    
    plt.title('Store Correlation Matrix (Sorted by Cluster)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Calculate average within-cluster and between-cluster correlations
    print("\nCorrelation Analysis:")
    print("=" * 60)
    
    for cluster_id in range(optimal_k):
        cluster_stores = hourly_features[hourly_features['cluster'] == cluster_id].index
        
        if len(cluster_stores) > 1:
            # Within-cluster correlation
            cluster_corr = correlation_matrix.loc[cluster_stores, cluster_stores]
            # Exclude diagonal
            mask = np.triu(np.ones_like(cluster_corr, dtype=bool), k=1)
            within_corr = cluster_corr.where(mask).stack().mean()
            
            print(f"Cluster {cluster_id}: Average within-cluster correlation = {within_corr:.3f}")
    
    print("\nHigh correlation within clusters indicates similar popular times patterns.")

## 9. Summary and Recommendations

In [ ]:
# Generate summary report
if df is not None:
    print("="*80)
    print("POPULAR TIMES CLUSTERING ANALYSIS - SUMMARY REPORT")
    print("="*80)
    
    print(f"\n1. DATA OVERVIEW:")
    print(f"   • Total stores analyzed: {len(hourly_features)}")
    print(f"   • Data points per store: {len(df) // len(hourly_features)}")
    print(f"   • Days covered: {', '.join(df['Day'].unique())}")
    
    print(f"\n2. CLUSTERING RESULTS:")
    print(f"   • Optimal number of trends: {optimal_k}")
    print(f"   • Silhouette score: {silhouette_results['optimal_score']:.4f}")
    print(f"   • Cluster quality: ", end="")
    
    if silhouette_results['optimal_score'] > 0.5:
        print("Strong ✓")
    elif silhouette_results['optimal_score'] > 0.3:
        print("Moderate")
    else:
        print("Weak")
    
    print(f"\n3. CLUSTER DISTRIBUTION:")
    for cluster_id in range(optimal_k):
        count = (final_labels == cluster_id).sum()
        percentage = count / len(final_labels) * 100
        print(f"   • Cluster {cluster_id}: {count} stores ({percentage:.1f}%)")
    
    print(f"\n4. KEY INSIGHTS:")
    print(f"   • There are {optimal_k} distinct popular times trends in your data")
    print(f"   • Stores in the same cluster show similar hourly traffic patterns")
    print(f"   • This can help with:")
    print(f"     - Staffing optimization")
    print(f"     - Inventory management")
    print(f"     - Marketing campaign timing")
    print(f"     - Store grouping for operational decisions")
    
    print("\n" + "="*80)
    
    # Save cluster assignments
    cluster_assignments = pd.DataFrame({
        'Reference_Id': hourly_features.index,
        'Location': [location_map.get(sid, 'Unknown') for sid in hourly_features.index],
        'Cluster': final_labels
    })
    
    output_path = os.path.join(project_root, 'data', 'cluster_assignments.csv')
    cluster_assignments.to_csv(output_path, index=False)
    print(f"\n✓ Cluster assignments saved to: {output_path}")

## 10. Next Steps

To use this notebook:
1. Create a CSV file at `data/popular_times.csv` with your data
2. The CSV should have columns: `hour`, `Reference_Id`, `Location`, `Day`, `Ratio`, `Adjusted_ratio`
3. Run all cells to perform the analysis
4. Review the optimal number of clusters and patterns
5. Use the cluster assignments for operational decisions